In [2]:
# Cell 1: 环境准备
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.engineering_intelligence.technology_brief import generate_technology_brief
from src.engineering_intelligence.strategy_classifier import get_all_strategies

driver = get_driver()
print("✅ 模块导入成功")

✅ 模块导入成功


In [2]:
# Cell 2: 确保 Stage2 demo 数据（幂等）
from src.engineering_intelligence.strategy_classifier import create_engineering_strategy
from src.engineering_intelligence.construct_service import create_engineered_construct
from src.engineering_intelligence.claim_extractor import (
    create_technical_claim, create_technical_result,
)
from src.engineering_intelligence.technology_assessment import create_technology_assessment
from src.shared.pathogen_service import get_or_create_pathogen

# --- 5 类策略 ---
STRATEGY_DEFS = [
    ("host_range_expansion",   "通过改造尾纤维或受体结合蛋白扩展宿主范围"),
    ("tail_fiber_engineering", "尾纤维定向改造，改变受体识别特异性"),
    ("lysis_enhancement",      "改造裂解模块或裂解酶，增强裂解活性"),
    ("biofilm_disruption",     "引入降解胞外基质的酶或改造穿透能力"),
    ("payload_delivery",       "递送抗耐药 / 抗毒力 / CRISPR 等外源载荷"),
]
strategy_ids = {}
for stype, desc in STRATEGY_DEFS:
    strategy_ids[stype] = create_engineering_strategy(
        driver, strategy_type=stype, description=desc,
        evidence_maturity="in_vitro", actor_id="demo",
    )
print(f"✅ 策略：{len(strategy_ids)} 个")

# --- 病原体 ---
for sp in ["Klebsiella pneumoniae", "Pseudomonas aeruginosa",
           "Staphylococcus aureus", "Escherichia coli"]:
    get_or_create_pathogen(driver, species=sp, pathogen_type="bacteria", actor_id="demo")
print("✅ 病原体就绪")

# --- 10 个构建体 ---
CONSTRUCT_DEFS = [
    ("host_range_expansion",   "vB_Kpn_HRE_001", "Klebsiella pneumoniae", "in_vitro_tested", "2024-06-10"),
    ("host_range_expansion",   "vB_Pae_HRE_002", "Pseudomonas aeruginosa","in_vitro_tested", "2024-09-01"),
    ("tail_fiber_engineering", "vB_Kpn_TFE_003", "Klebsiella pneumoniae", "in_vitro_tested", "2024-06-10"),
    ("tail_fiber_engineering", "vB_Eco_TFE_004", "Escherichia coli",      "proposed",        "2025-01-10"),
    ("lysis_enhancement",      "vB_Pae_LYE_005", "Pseudomonas aeruginosa","in_vitro_tested", "2024-09-01"),
    ("lysis_enhancement",      "vB_Sau_LYE_006", "Staphylococcus aureus", "in_vitro_tested", "2024-09-01"),
    ("biofilm_disruption",     "vB_Pae_BFD_007", "Pseudomonas aeruginosa","in_vitro_tested", "2024-09-01"),
    ("biofilm_disruption",     "vB_Sau_BFD_008", "Staphylococcus aureus", "proposed",        "2024-09-01"),
    ("payload_delivery",       "vB_Kpn_PLD_009", "Klebsiella pneumoniae", "in_vitro_tested", "2024-05-20"),
    ("payload_delivery",       "vB_Pae_PLD_010", "Pseudomonas aeruginosa","proposed",        "2024-05-20"),
]

def _get_construct_id(name):
    with driver.session() as s:
        r = s.run("MATCH (ec:EngineeredPhageConstruct {public_name: $n}) "
                  "RETURN ec.construct_id AS id LIMIT 1", n=name).single()
        return r["id"] if r else None

construct_ids = {}
for stype, name, target_sp, status, pub_date in CONSTRUCT_DEFS:
    existing = _get_construct_id(name)
    if existing:
        construct_ids[name] = (existing, stype)
        continue
    cid = create_engineered_construct(
        driver, public_name=name, construct_code=name.upper(),
        parent_phage_name=None, intended_effects=[stype],
        target_pathogen_species=[target_sp],
        strategy_ids=[strategy_ids[stype]],
        construct_status=status, first_public_date=pub_date, actor_id="demo",
    )
    construct_ids[name] = (cid, stype)
print(f"✅ 构建体：{len(construct_ids)} 个")

# --- 主张 + 结果 + 评估 ---
CLAIM_TYPE_MAP = {
    "host_range_expansion": "host_range",
    "tail_fiber_engineering": "host_range",
    "lysis_enhancement": "efficacy",
    "biofilm_disruption": "efficacy",
    "payload_delivery": "safety",
}
for name, (cid, stype) in construct_ids.items():
    with driver.session() as s:
        existing = s.run("MATCH (tr:TechnicalResult)-[:RESULT_FOR]->"
                         "(ec:EngineeredPhageConstruct {construct_id: $cid}) "
                         "RETURN count(tr) AS c", cid=cid).single()["c"]
    if existing:
        continue
    ct = CLAIM_TYPE_MAP[stype]
    create_technical_claim(
        driver, claim_type=ct,
        claim_text=f"{name} 在体外展示出改进的 {ct} 表型",
        claimant_type="publication", evidence_context="in_vitro",
        construct_id=cid, strategy_id=strategy_ids[stype], actor_id="demo",
    )
    create_technical_result(
        driver, result_type=ct, study_context="in_vitro",
        outcome_direction="positive", metric_name="relative_improvement",
        metric_value=2.0, metric_unit="fold", comparator="亲本噬菌体",
        sample_size=10, limitation_summary="仅体外数据，需要体内验证",
        reproducibility_status="single_source", construct_id=cid, actor_id="demo",
    )
    create_technology_assessment(
        driver, subject_type="construct", subject_id=cid,
        evidence_maturity="in_vitro", technical_relevance="high",
        translational_potential="medium", manufacturability_risk="medium",
        safety_uncertainty="low", ip_relevance="medium",
        internal_capability_gap="high" if stype in ("payload_delivery", "biofilm_disruption") else "medium",
        assessment_summary=f"{name} 的体外证据支持其技术相关性，转化与可制造性仍需验证。",
        actor_id="demo",
    )
print("✅ 主张 / 结果 / 评估 就绪")

# --- 策略级评估 ---
for stype, sid in strategy_ids.items():
    with driver.session() as s:
        existing = s.run("MATCH (ta:TechnologyAssessment {subject_type: 'strategy', subject_id: $sid}) "
                         "RETURN count(ta) AS c", sid=sid).single()["c"]
    if existing:
        continue
    create_technology_assessment(
        driver, subject_type="strategy", subject_id=sid,
        evidence_maturity="in_vitro", technical_relevance="high",
        translational_potential="medium", manufacturability_risk="medium",
        safety_uncertainty="medium", ip_relevance="medium",
        internal_capability_gap="medium",
        assessment_summary=f"{stype} 策略公开证据以体外为主，需要更多转化证据。",
        actor_id="demo",
    )
print("✅ 策略级评估就绪")
print("\n🎯 Cell 2 完成")

✅ 策略：5 个
✅ 病原体就绪
✅ 构建体：10 个
✅ 主张 / 结果 / 评估 就绪
✅ 策略级评估就绪

🎯 Cell 2 完成


In [3]:
# Cell 3: 为 5 类策略各生成 1 份技术简报
strategies = get_all_strategies(driver)
print(f"当前策略数：{len(strategies)}\n")

brief_ids = []
for s in strategies:
    stype = s["strategy_type"]
    brief = generate_technology_brief(
        driver,
        strategy_id=s["id"],
        technology_question=f"{stype} 的公开证据成熟度如何？",
        persist=True,
    )
    brief_ids.append(brief["brief_id"])
    print(f"📄 {stype}")
    print(f"   brief_id   : {brief['brief_id']}")
    print(f"   构建体     : {len(brief['known_constructs'])}")
    print(f"   主张/结果  : {len(brief['technical_claims'])} / {len(brief['technical_results'])}")
    print(f"   最高成熟度 : {brief['evidence_maturity']['highest_maturity']}")
    print(f"   建议下一步 :")
    for step in brief["recommended_next_steps"]:
        print(f"      - {step}")
    print()

print(f"✅ 共生成 {len(brief_ids)} 份技术简报")

当前策略数：5

📄 biofilm_disruption
   brief_id   : CI:BRIEF:A2E4D1DB
   构建体     : 2
   主张/结果  : 2 / 2
   最高成熟度 : conceptual
   建议下一步 :
      - 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。

📄 host_range_expansion
   brief_id   : CI:BRIEF:11ABC6EF
   构建体     : 2
   主张/结果  : 2 / 2
   最高成熟度 : conceptual
   建议下一步 :
      - 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。

📄 lysis_enhancement
   brief_id   : CI:BRIEF:325517E6
   构建体     : 2
   主张/结果  : 2 / 2
   最高成熟度 : conceptual
   建议下一步 :
      - 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。

📄 payload_delivery
   brief_id   : CI:BRIEF:54E0A2D4
   构建体     : 2
   主张/结果  : 2 / 2
   最高成熟度 : conceptual
   建议下一步 :
      - 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。

📄 tail_fiber_engineering
   brief_id   : CI:BRIEF:7C8AD329
   构建体     : 2
   主张/结果  : 2 / 2
   最高成熟度 : conceptual
   建议下一步 :
      - 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。

✅ 共生成 5 份技术简报


In [4]:
# Cell 4: 落库校验
from src.ci.intelligence_product_service import get_intelligence_product

print("=" * 60)
print("落库校验：engineered_phage_technology 简报")
print("=" * 60)

with driver.session() as s:
    rows = list(s.run("""
        MATCH (b:IntelligenceProduct {brief_type: 'engineered_phage_technology'})
        RETURN b.brief_id AS id, b.as_of_date AS as_of,
               b.review_status AS status, b.created_at AS created_at
        ORDER BY b.created_at DESC
    """))

print(f"共 {len(rows)} 份技术简报\n")
for r in rows:
    with driver.session() as s2:
        covers = list(s2.run("""
            MATCH (b:IntelligenceProduct {brief_id: $bid})-[:COVERS]->(x)
            RETURN labels(x)[0] AS label, count(*) AS cnt
        """, bid=r["id"]))
    cov = {x["label"]: x["cnt"] for x in covers}
    print(f"  {r['id']}  as_of={r['as_of']}  status={r['status']}  covers={cov}")

print()
if len(rows) >= 5:
    print("🎉 MVP 验收通过：技术简报数 ≥ 5")
else:
    print(f"⚠️ 当前 {len(rows)} 份，未达 5 份验收要求")

落库校验：engineered_phage_technology 简报
共 5 份技术简报

  CI:BRIEF:7C8AD329  as_of=2026-09-27  status=pending  covers={'Pathogen': 4, 'EngineeredPhageConstruct': 2, 'EngineeringStrategy': 1}
  CI:BRIEF:54E0A2D4  as_of=2026-09-27  status=pending  covers={'Pathogen': 4, 'EngineeredPhageConstruct': 2, 'EngineeringStrategy': 1}
  CI:BRIEF:325517E6  as_of=2026-09-27  status=pending  covers={'Pathogen': 4, 'EngineeredPhageConstruct': 2, 'EngineeringStrategy': 1}
  CI:BRIEF:11ABC6EF  as_of=2026-09-27  status=pending  covers={'Pathogen': 4, 'EngineeredPhageConstruct': 2, 'EngineeringStrategy': 1}
  CI:BRIEF:A2E4D1DB  as_of=2026-09-27  status=pending  covers={'Pathogen': 4, 'EngineeredPhageConstruct': 2, 'EngineeringStrategy': 1}

🎉 MVP 验收通过：技术简报数 ≥ 5


In [3]:
# 验证 Cell：修复后的证据成熟度聚合
import importlib
import src.engineering_intelligence.technology_brief as tb
importlib.reload(tb)
from src.engineering_intelligence.technology_brief import generate_technology_brief

strategies = get_all_strategies(driver)
print(f"当前策略数：{len(strategies)}\n")

# 只取第一个策略验证（避免生成 5 份简报）
target = strategies[0]
print(f"验证策略：{target['strategy_type']} ({target['id']})\n")

brief = generate_technology_brief(
    driver,
    strategy_id=target["id"],
    persist=True,
)

print("=" * 60)
print(f"brief_id     : {brief['brief_id']}")
print(f"最高成熟度   : {brief['evidence_maturity']['highest_maturity']}")
print(f"成熟度分布   : {brief['evidence_maturity']['distribution']}")
print(f"构建体级成熟度({len(brief['evidence_maturity']['by_construct'])} 个):")
for c in brief["evidence_maturity"]["by_construct"]:
    print(f"  {c['name']}: {c['maturity']}")

print(f"\n内部能力差距({len(brief['internal_capability_gaps'])} 条):")
for g in brief["internal_capability_gaps"]:
    print(f"  [{g['gap_level']}] {g['subject_type']} {g['subject_id']}")

print(f"\n安全性风险({len(brief['safety_uncertainties'])} 条):")
for s in brief["safety_uncertainties"]:
    print(f"  [{s['level']}] {s['subject_type']} {s['subject_id']}")
print("=" * 60)

当前策略数：5

验证策略：biofilm_disruption (ENG:STRAT:B7FD45B5)

brief_id     : CI:BRIEF:895793D3
最高成熟度   : in_vitro
成熟度分布   : {'conceptual': 0, 'in_vitro': 2, 'in_vivo': 0, 'clinical': 0}
构建体级成熟度(2 个):
  vB_Sau_BFD_008: in_vitro
  vB_Pae_BFD_007: in_vitro

内部能力差距(3 条):
  [medium] strategy ENG:STRAT:B7FD45B5
  [high] construct ENG:CONST:E2ED2755
  [high] construct ENG:CONST:2B0A0785

安全性风险(3 条):
  [medium] strategy ENG:STRAT:B7FD45B5
  [low] construct ENG:CONST:E2ED2755
  [low] construct ENG:CONST:2B0A0785
